<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/Step05bEmbedFull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 05b · Fingerprint the crops: full, all videos

Turns the crops of **Step 04b** (`Step04bCropFull`) into **fingerprints**: 384 numbers per crop from DINOv3 ViT-S/16 describing coat pattern and shape. Two crops of the same cow should have similar fingerprints (cosine similarity near 1); that is what Re-ID relies on.

**Several videos in one run.** With `VideoStems = "AllNew"` it fingerprints **every video folder** in `CalvingPenPipelineOutputs/` that has `Step04bCropFull` but no up-to-date `Step05bEmbedFull` yet (new videos, or videos whose crops were redone), ~1–2 min per video. `"All"` redoes all of them; a list of folder names does only those. Typical use: teammates run `PipelineSteps` up to step 04 for their videos; one person then runs this notebook and Step 06b.

**Only SAM outputs:** videos whose crops did not come from SAM 2.1 tracks (step 03) are skipped (`OnlySam = True`). The table before and after the run shows, per video, which steps are done, where the crops came from, the model, and whether the video is in the pool.

**DINOv3 needs access:** request it at huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m, then add a read token as the Colab secret `HF_TOKEN` (steps in `CalvingPen/README.md`). Here there is **no fallback** to DINOv2: if DINOv3 cannot be loaded it stops, so the pool never mixes models.

Writes into each video's folder, `CalvingPenPipelineOutputs/<video>/Step05bEmbedFull/`, **overwriting**: `Embeddings.npy` + `EmbeddingIndex.csv` (input for step 06b), `TrackSimilarity.csv` / `.png` (how alike 5-minute pieces of each track are), `Model.txt`, `Source.json` (which crops they belong to).

Needs a **T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install the model libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Embed import EmbedVideos
from CalvingPipeline.Pipeline import VideoStatus
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "not set - DINOv2 fallback will be used")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStems = "AllNew"    # "AllNew" = every video with crops but no up-to-date fingerprints; "All" = redo all;
                         # or a list of output folder names, e.g. ["2026-10-29 Calving Pen 2 (3)"]
OnlySam = True           # skip videos whose crops did not come from SAM 2.1 tracks (step 03)
ModelId = "facebook/dinov3-vits16-pretrain-lvd1689m"
GroupSeconds = 300       # heatmap: split long tracks into 5-min pieces (T1@15m = track 1 from minute 15)

import pandas as pd
from IPython.display import display
display(VideoStatus(OutputRoot))   # before: what each video folder has

In [ ]:
# 3. Run: fingerprint the selected videos, then show the status and each video's heatmap.
from IPython.display import Image

Done = EmbedVideos(OutputRoot, VideoStems, ModelId=ModelId, Token=HfToken, GroupSeconds=GroupSeconds, OnlySam=OnlySam)
display(Done)
display(VideoStatus(OutputRoot))   # after
for Stem in (Done.VideoStem[Done.Result == "done"] if len(Done) else []):
    print(Stem)
    display(Image(str(OutputRoot / Stem / "Step05bEmbedFull" / "TrackSimilarity.png")))

## About this step

**Model:** **DINOv3 ViT-S/16** (Meta, `facebook/dinov3-vits16-pretrain-lvd1689m`, gated: needs `HF_TOKEN`); fallback **DINOv2-small**. Each crop (padded to a square with the mask grey) becomes 384 numbers; cosine similarity compares two crops (≈1 = look the same).

**How to read the heatmap:** each square is the average similarity between the crops of two track pieces; bright blocks = pieces that look alike.

**What we found (earlier, 306 crops from the ByteTrack tracks, one cow):** the fingerprints group by **pose**, not by animal: lying crops 0.76–0.86 with each other, standing / rear 0.74–0.84, but lying vs. standing only 0.57–0.62 although it is the same cow. Re-ID must therefore compare a new crop with her **closest** crops, not an average. Different cows need to score below ~0.55; that needs a second cow to check. To be redone on the SAM crops.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.